# ADAC Workshop 2 26L: Distributed SparkML on Managed Service for Apache Spark

In this lab you run a **real distributed machine-learning pipeline** on the Microsoft Security Incident (GUIDE) dataset from Lab 1 — this time on **Google Managed Service for Apache Spark** (Dataproc Serverless, Lightning engine), from a free Colab front-end via **Spark Connect**, on **your own** GCP project funded by your education coupon.

**Two things you will learn to see:**
1. **The pipeline stages** — both the SparkML `Pipeline` stages (Transformers vs Estimators, lazy `fit`/`transform`) and the Spark execution-DAG stages the distributed `.fit()` triggers.
2. **How it scales** — run the same pipeline on growing data and measure throughput.

> ⚠️ Cost: you pay serverless compute from your coupon. Use **2 executors**, and **delete your session at the end** (Part 4).

## Part 0 — Get the data, provision your cluster, connect

### 0.1 Download the dataset (KaggleHub, like Lab 1)

In [ ]:
%pip install -q kagglehub google-cloud-storage dataproc-spark-connect
import kagglehub, glob, os
path = kagglehub.dataset_download('Microsoft/microsoft-security-incident-prediction')
train_csv = glob.glob(os.path.join(path, '**', 'GUIDE_Train.csv'), recursive=True)[0]
test_csv  = glob.glob(os.path.join(path, '**', 'GUIDE_Test.csv'),  recursive=True)[0]
print(train_csv, test_csv)

### 0.2 Authenticate to *your* GCP project and create a bucket

Set `PROJECT_ID` to your coupon project. The bucket must be in the **same region** as the Spark session (`europe-west1`).

In [ ]:
from google.colab import auth
auth.authenticate_user()
PROJECT_ID = ''  # <-- your coupon project id
REGION = 'europe-west1'
SESSION_ID = 'adac-lab2'  # lowercase letters/digits/hyphens
BUCKET = f'{PROJECT_ID}-adac-lab2'
!gcloud config set project {PROJECT_ID}
!gsutil mb -l {REGION} gs://{BUCKET} || echo 'bucket may already exist'

In [ ]:
# Upload both CSVs to your bucket so the serverless cluster can read them
!gsutil -m cp '{test_csv}'  gs://{BUCKET}/guide/GUIDE_Test.csv
!gsutil -m cp '{train_csv}' gs://{BUCKET}/guide/GUIDE_Train.csv
TRAIN = f'gs://{BUCKET}/guide/GUIDE_Train.csv'
TEST  = f'gs://{BUCKET}/guide/GUIDE_Test.csv'

### 0.3 Provision the serverless Spark session — *wyklikać* (click it out)

Create an **interactive Spark Connect session** with the **Lightning engine** (premium tier), runtime **2.3**, and **2 executors**. Either click it out in the console, or run the gcloud command — both produce a session named by `SESSION_ID`.

**Console (wyklikać):** Dataproc → Serverless → Interactive Sessions → **Create**; Region `europe-west1`; Runtime version `2.3`; **Premium** tier (enables Lightning Engine); session type **Spark Connect**; under Properties add `spark.executor.instances=2`. Submit and wait for **Active**. *(Teacher: drop console screenshots here.)*

**gcloud (equivalent):**
```bash
gcloud beta dataproc sessions create spark $SESSION_ID \
  --project=$PROJECT_ID --location=$REGION --version=2.3 \
  --properties=dataproc.tier=premium,spark.dataproc.engine=lightningEngine,spark.executor.instances=2
```

> Note: with the `dataproc-spark-connect` client there is **no `sc://` endpoint to copy** — the client resolves the session from your project + region + `SESSION_ID`.

### 0.4 Connect from Colab via Spark Connect

In [ ]:
# TODO 0.4: connect to YOUR serverless session via Spark Connect.
# Use DataprocSparkSession.builder with: projectId(PROJECT_ID), location(REGION),
# dataprocSessionId(SESSION_ID), runtimeVersion('2.3'), and configs enabling the
# Lightning engine (dataproc.tier=premium, spark.dataproc.engine=lightningEngine)
# and 2 executors (spark.executor.instances=2). Finish with print(spark.range(5).count()).
spark = ...  # <-- your code here

## Part 1 — Explore the data and build the SparkML pipeline

### 1.1 Distributed read + a first look
We read the CSV on the cluster and drop the ~0.5% of rows with no `IncidentGrade` label.

In [ ]:
from pyspark.sql import functions as F
raw = (spark.read.option('header', True).option('inferSchema', True).csv(TRAIN))
raw = raw.where(F.col('IncidentGrade').isNotNull())
print('rows:', raw.count())
raw.groupBy('IncidentGrade').count().show()

### 1.2 The pipeline, stage by stage

A SparkML `Pipeline` is an ordered list of **stages**. Each stage is either a **Transformer** (`transform`: DataFrame → DataFrame, e.g. `OneHotEncoder`) or an **Estimator** (`fit`: DataFrame → Model, e.g. `StringIndexer`, `RandomForestClassifier`). `Pipeline.fit()` runs the stages in order and returns a `PipelineModel`. Nothing is computed until an action triggers it — that is when Spark builds a **DAG of execution stages** across the executors.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

CAT_COLS = ['Category', 'EntityType', 'EvidenceRole', 'CountryCode']

# TODO 1.2: build the pipeline stages, in order:
#   1. StringIndexer on 'IncidentGrade' -> 'label' (handleInvalid='skip')
#   2. one StringIndexer per CAT_COLS column -> '<col>_idx' (handleInvalid='keep')
#   3. OneHotEncoder over the '_idx' columns -> '_oh' columns
#   4. VectorAssembler of the '_oh' columns -> 'features'
#   5. RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=40)
# Then: pipeline = Pipeline(stages=[...]) and print the stage types.
pipeline = ...  # <-- your code here

## Part 2 — Fit, evaluate, and read the distributed execution

### 2.1 Fit on Train, evaluate on Test
`fit` is where the distributed work happens. We cache the inputs so the fit time is not dominated by re-reading from GCS.

In [ ]:
import time
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# TODO 2.1: cache train (raw) and test (read TEST, drop null IncidentGrade);
# time pipeline.fit(train) -> model; transform test; print F1 and accuracy with
# MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction').
model = ...  # <-- your code here

### 2.2 Confusion matrix + feature importances
Inspect *what* the model learned.

In [ ]:
pred.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()
rf_model = model.stages[-1]
print('feature importances vector length:', rf_model.featureImportances.size)

### 2.3 Read the Spark UI

Open the **Spark UI** for your session (Dataproc → Serverless → Interactive Sessions → your session → View Spark UI). Find the job triggered by `fit`, open its **DAG**, and note: how many **stages**, where the **shuffle** boundaries are, and how tasks were split across your **2 executors**. Record your observations in the answer cell in Part 4.

## Part 3 — How does it scale?

**Weak scaling:** run the *same* pipeline on growing input and watch wall-clock and throughput (rows/sec). We sweep fractions of Train; a warm-up run excludes catalog/JIT/IO warm-up. Executors stay fixed at **2**.

In [ ]:
import time
# TODO 3: do one warm-up fit (not measured). Then for fr in [0.1,0.25,0.5,1.0]:
#   sample train at fr, cache+count rows, time pipeline.fit, record rows & seconds,
#   and print rows + fit-time + throughput (rows/sec).
rows, secs = [], []  # <-- your code here

### 3.1 Plot time and throughput vs data volume

In [ ]:
import matplotlib.pyplot as plt
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(rows, secs, 'o-'); a.set_xlabel('rows'); a.set_ylabel('fit time (s)'); a.set_title('Scaling: time')
thr = [n/s for n, s in zip(rows, secs)]
b.plot(rows, thr, 'o-'); b.set_xlabel('rows'); b.set_ylabel('rows/sec'); b.set_title('Scaling: throughput')
plt.tight_layout(); plt.show()

### 3.2 (Optional bonus) 4 executors

If your coupon allows: **delete this session**, create a new one with `spark.executor.instances=4`, re-connect (Part 0.4), and re-run the `frac=1.0` point. Compare fit time to your 2-executor result — a peek at *strong* scaling.

## Part 4 — Conclusions & teardown

### 4.1 Your answers
Fill these in; they are part of your submission.

In [ ]:
from IPython.display import Markdown, display
PROMPT = 'How many Spark stages did `fit` create, and where was the shuffle?'
ANSWER = ''  # <-- write your answer here
display(Markdown('**' + PROMPT + '**\n\n' + ANSWER))

In [ ]:
from IPython.display import Markdown, display
PROMPT = 'From your scaling plot: is throughput (rows/sec) roughly flat as data grows? What does that imply?'
ANSWER = ''  # <-- write your answer here
display(Markdown('**' + PROMPT + '**\n\n' + ANSWER))

In [ ]:
from IPython.display import Markdown, display
PROMPT = 'What did the 2 executors give you vs single-node pandas in Lab 1?'
ANSWER = ''  # <-- write your answer here
display(Markdown('**' + PROMPT + '**\n\n' + ANSWER))

### 4.2 Delete your session (save your coupon!)
Delete the interactive session in the console (Dataproc → Serverless → Interactive Sessions → your session → Terminate), or run the teardown cell below.

In [ ]:
try:
    spark.stop()
except Exception as e:
    print('stop:', e)
# Also terminate the session in the console so it stops billing.